Note: the basis for this code was given by James Day in MATLAB.

The goal of this code is to mask the sinogram for dead or unstable pixels. The projection data is also obtained by logarithmic normalization of the transmitted beam intensity and the beam intensity from the air scan.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

In [ ]:
import os
import numpy as np

In [ ]:
# Source: MATLAB code provided by James Day
def get_local_and_val(image, map_array, mask, k_idx, row_slice, col_slice, mask_slice):
  local_map = map_array[k_idx, row_slice, col_slice]
  local_image = image[k_idx, row_slice, col_slice]

  local_mask = mask[mask_slice]

  local = np.abs(local_map-1)*local_mask

  sum_local = np.sum(local)

  if sum_local > 0:
    weighted_sum = np.sum(local_image*local)
    val = weighted_sum/sum_local
  else:
    val = 0

  return [local, val]


def put_on_mask_3d(image, image_mask, mask_array):
  angles, rows, cols = image.shape
  rows_mask, cols_mask = mask_array.shape
  flag = False

  output_image = image.copy()
  map_array = image_mask[:angles, :, :]

  while (np.sum(map_array) > 0) and not flag:
    if np.sum(map_array) == 0:
      flag = True

    elif np.sum(map_array) > (angles*rows*cols)/2:
      flag = True

    else:
      for k in range(angles):
        if np.sum(output_image[k,:,:]) == 0:
          if k == 0:
            output_image[k,:,:] = output_image[k+1, :, :]
          elif k == angles-1:
            output_image[k,:,:] = output_image[k-1, :, :]
          else:
            output_image[k,:,:] = (output_image[k+1,:,:] + output_image[k-1,:,:])/2

        for i in range(rows):
          for j in range(cols):
            if map_array[k,i,j] == 1:
              is_corner = (i < 2 and j < 2) or (i < 2 and j>= cols-2) or (i >= rows-2 and j >= cols-2) or (i >= rows-2 and j < 2)
              is_edge = not is_corner and (i<2 or j<2 or i>= rows-2 or j>= cols-2)

              if is_corner:
                # top left corner
                if i < 2 and j < 2:
                  if i == 0 and j == 0:
                      row_slice = slice(i, i + 3)
                      col_slice = slice(j, j + 3)
                      mask_slice = (slice(2, 5), slice(2, 5))
                  elif i == 0 and j == 1:
                      row_slice = slice(i, i + 3)
                      col_slice = slice(j - 1, j + 3)
                      mask_slice = (slice(2, 5), slice(1, 5))
                  elif i == 1 and j == 0:
                      row_slice = slice(i - 1, i + 3)
                      col_slice = slice(j, j + 3)
                      mask_slice = (slice(1, 5), slice(2, 5))
                  elif i == 1 and j == 1:
                      row_slice = slice(i - 1, i + 3)
                      col_slice = slice(j - 1, j + 3)
                      mask_slice = (slice(1, 5), slice(1, 5))

                # top right corner
                elif i < 2 and j >= cols - 2:
                  if i == 0 and j == cols - 1:
                      row_slice = slice(i, i + 3)
                      col_slice = slice(cols - 3, cols)
                      mask_slice = (slice(2, 5), slice(0, 3))
                  elif i == 0 and j == cols - 2:
                      row_slice = slice(i, i + 3)
                      col_slice = slice(cols - 4, cols)
                      mask_slice = (slice(2, 5), slice(0, 4))
                  elif i == 1 and j == cols - 1:
                      row_slice = slice(i - 1, i + 3)
                      col_slice = slice(cols - 3, cols)
                      mask_slice = (slice(1, 5), slice(0, 3))
                  elif i == 1 and j == cols - 2:
                      row_slice = slice(i - 1, i + 3)
                      col_slice = slice(cols - 4, cols)
                      mask_slice = (slice(1, 5), slice(0, 4))

                # bottom right corner
                elif i >= rows - 2 and j >= cols - 2:
                  if i == rows - 1 and j == cols - 1:
                      row_slice = slice(rows - 3, rows)
                      col_slice = slice(cols - 3, cols)
                      mask_slice = (slice(0, 3), slice(0, 3))
                  elif i == rows - 1 and j == cols - 2:
                      row_slice = slice(rows - 3, rows)
                      col_slice = slice(cols - 4, cols)
                      mask_slice = (slice(0, 3), slice(0, 4))
                  elif i == rows - 2 and j == cols - 1:
                      row_slice = slice(rows - 4, rows)
                      col_slice = slice(cols - 3, cols)
                      mask_slice = (slice(0, 4), slice(0, 3))
                  elif i == rows - 2 and j == cols - 2:
                      row_slice = slice(rows - 4, rows)
                      col_slice = slice(cols - 4, cols)
                      mask_slice = (slice(0, 4), slice(0, 4))

                # bottom left corner
                elif i >= rows - 2 and j < 2:
                  if i == rows - 1 and j == 0:
                      row_slice = slice(rows - 3, rows)
                      col_slice = slice(j, j + 3)
                      mask_slice = (slice(0, 3), slice(2, 5))
                  elif i == rows - 1 and j == 1:
                      row_slice = slice(rows - 3, rows)
                      col_slice = slice(j - 1, j + 3)
                      mask_slice = (slice(0, 3), slice(1, 5))
                  elif i == rows - 2 and j == 0:
                      row_slice = slice(rows - 4, rows)
                      col_slice = slice(j, j + 3)
                      mask_slice = (slice(0, 4), slice(2, 5))
                  elif i == rows - 2 and j == 1:
                      row_slice = slice(rows - 4, rows)
                      col_slice = slice(j - 1, j + 3)
                      mask_slice = (slice(0, 4), slice(1, 5))

                # value for corner
                local, val = get_local_and_val(output_image, map_array, mask_array, k, row_slice, col_slice, mask_slice)

              elif is_edge:
                # top edge
                if i < 2:
                  if i == 0:
                      row_slice = slice(i, i + 3)
                      col_slice = slice(j - 2, j + 3)
                      mask_slice = (slice(2, 5), slice(0, 5))
                  elif i == 1:
                      row_slice = slice(i - 1, i + 3)
                      col_slice = slice(j - 2, j + 3)
                      mask_slice = (slice(1, 5), slice(0, 5))

                # left edge
                elif j < 2:
                  if j == 0:
                      row_slice = slice(i - 2, i + 3)
                      col_slice = slice(j, j + 3)
                      mask_slice = (slice(0, 5), slice(2, 5))
                  elif j == 1:
                      row_slice = slice(i - 2, i + 3)
                      col_slice = slice(j - 1, j + 3)
                      mask_slice = (slice(0, 5), slice(1, 5))

                # bottom edge
                elif i >= rows - 2:
                  if i == rows - 1:
                      row_slice = slice(rows - 3, rows)
                      col_slice = slice(j - 2, j + 3)
                      mask_slice = (slice(0, 3), slice(0, 5))
                  elif i == rows - 2:
                      row_slice = slice(rows - 4, rows)
                      col_slice = slice(j - 2, j + 3)
                      mask_slice = (slice(0, 4), slice(0, 5))

                # right edge
                elif j >= cols - 2:
                  if j == cols - 1:
                      row_slice = slice(i - 2, i + 3)
                      col_slice = slice(cols - 3, cols)
                      mask_slice = (slice(0, 5), slice(0, 3))
                  elif j == cols - 2:
                      row_slice = slice(i - 2, i + 3)
                      col_slice = slice(cols - 4, cols)
                      mask_slice = (slice(0, 5), slice(0, 4))

                # value for edge
                local, val = get_local_and_val(output_image, map_array, mask_array, k, row_slice, col_slice, mask_slice)

              else:
                row_slice = slice(i - 2, i + 3)
                col_slice = slice(j - 2, j + 3)
                mask_slice = (slice(0, 5), slice(0, 5))

                local, val = get_local_and_val(output_image, map_array, mask_array, k, row_slice, col_slice, mask_slice)

              output_image[k, i, j] = val

              if np.isnan(val) or np.isinf(val) or val == 0:
                  map_array[k, i, j] = 1
              else:
                  map_array[k, i, j] = 0

  return output_image

In [ ]:
# Source: MATLAB code provided by James Day
def map_mask_zeros(image):
  mask_map = (
    np.isnan(image) |
    np.isinf(image) |
    (image == 0)
  ).astype(np.uint8)
  return mask_map

In [ ]:
# Source: MATLAB code provided by James Day
def process_image(image):
  angles, rows, cols = image.shape

  mask_array = np.zeros((5,5))

  for i in range(5):
    for j in range(5):
      if i == 2 and j == 2:
        mask_array[i, j] = 0
      else:
        mask_array[i, j] = 1/np.sqrt((i - 2)**2 + (j - 2)**2)

  image_mask = map_mask_zeros(image)

  gain_temp = put_on_mask_3d(image,image_mask,mask_array)
  gain_image = np.zeros((angles, rows, cols))

  for k in range(angles):
    gain_image[k,:,:] = gain_temp[k,:,:]/np.mean(gain_temp[k,:,:])

  return [image_mask, gain_image]

In [ ]:
def load_sinograms(base_filename, image_n):

  bin1_image = np.load(base_filename+f"Sinograms/{chr(image_n+65)}_bin1_image.npy")
  bin2_image = np.load(base_filename+f"Sinograms/{chr(image_n+65)}_bin2_image.npy")
  bin3_image = np.load(base_filename+f"Sinograms/{chr(image_n+65)}_bin3_image.npy")
  bin4_image = np.load(base_filename+f"Sinograms/{chr(image_n+65)}_bin4_image.npy")
  bin5_image = np.load(base_filename+f"Sinograms/{chr(image_n+65)}_bin5_image.npy")
  bin6_image = np.load(base_filename+f"Sinograms/{chr(image_n+65)}_bin6_image.npy")
  bin_all_image = np.load(base_filename+f"Sinograms/{chr(image_n+65)}_bin_all_image.npy")

  return [bin1_image, bin2_image, bin3_image, bin4_image, bin5_image, bin6_image, bin_all_image]

In [ ]:
# file names for objects to scan and air scans
FILENAME_BASE = '/content/gdrive/MyDrive/Honours_Thesis/3_SCANS/S1728_PigTail2_frames3600_res0.01_thresh25_35_45_55_65_75/'
FILENAME_BASE_AIR = "/content/gdrive/MyDrive/Honours_Thesis/3_SCANS/S1728_PigTail1_frames3600_res0.01_thresh25_35_45_55_65_75/"
FILENAME_BASE_BLANK = "/content/gdrive/MyDrive/Honours_Thesis/3_SCANS/S1728_PigTail2_frames3600_res0.01_thresh25_35_45_55_65_75/"

In [ ]:
os.makedirs(FILENAME_BASE+"Masked_Sinograms", exist_ok=True)

imb = 0 # image number of air scan
imm = 0 # image number of air scan

start_acquistion = 3 # starting image number to mask
end_aquisition = 3 # last image number to mask

bin1_air, bin2_air, bin3_air, bin4_air, bin5_air, bin6_air, bin_all_air = load_sinograms(FILENAME_BASE_AIR, imm)

[mask1,gain1] = process_image(bin1_air)
[mask2,gain2] = process_image(bin2_air)
[mask3,gain3] = process_image(bin3_air)
[mask4,gain4] = process_image(bin4_air)
[mask5,gain5] = process_image(bin5_air)
[mask6,gain6] = process_image(bin6_air)
[mask_all, gain_all] = process_image(bin_all_air)



---



In [ ]:
# Source: MATLAB code provided by James Day
def apply_corrections(image,mask_map,gain_map):
  angles, rows, cols = image.shape

  mask_array = np.zeros((5,5))

  for i in range(5):
    for j in range(5):
      if i == 2 and j == 2:
        mask_array[i, j] = 0
      else:
        mask_array[i, j] = 1/np.sqrt((i - 2)**2 + (j - 2)**2)

  image_out = put_on_mask_3d(image,mask_map,mask_array)

  gain_map_size = gain_map.shape[0]
  image_out_size = image_out.shape[0]

  if gain_map_size==image_out_size:
    image_out = image_out/gain_map
  else:
    cc = 1
    for ii in range(image_out_size):
      cc = (ii%gain_map_size)
      image_out[ii,:,:] = image_out[ii,:,:]/gain_map[cc,:,:]
      cc += 1


  return image_out



---



In [ ]:
# Source: MATLAB code provided by James Day
def map_mask_simple(image_bin, num_mode):
  maskmap = (
    np.isnan(image_bin) |
    np.isinf(image_bin) |
    (image_bin == 0)
  ).any(axis=0)

  value = np.abs(np.std(image_bin, axis=0, ddof=0) / np.mean(image_bin, axis=0)) * 100
  maskmap |= value > num_mode

  return maskmap.astype(np.uint8)

In [ ]:
# Source: MATLAB code provided by James Day
def get_local_and_val_2d_map(image_k_slice, map_array, mask, row_slice_map, col_slice_map, mask_slice):
  local_map = map_array[row_slice_map, col_slice_map]
  local_image = image_k_slice[row_slice_map, col_slice_map]

  local_mask = mask[mask_slice]
  local = np.abs(local_map - 1) * local_mask

  sum_local = np.sum(local)

  if sum_local > 0:
    weighted_sum = np.sum(local_image * local)
    val = weighted_sum / sum_local
  else:
    val = 0

  return [local, val]


def put_on_mask_2d(image, map_array, mask_array):
  angles, rows, cols = image.shape
  rows_mask, cols_mask = mask_array.shape

  output_image = image.copy()


  for k in range(angles):
    if np.sum(output_image) == 0:
      break

    if np.sum(map_array) > (192 * 72) / 2:
      break

    for i in range(rows):
      for j in range(cols):
        if map_array[i, j] == 1 and np.sum(output_image[k, :, :]) > 0:

          row_slice_map = None
          col_slice_map = None
          mask_slice = None

          is_corner = (i < 2 and j < 2) or (i < 2 and j >= cols - 2) or \
                      (i >= rows - 2 and j >= cols - 2) or (i >= rows - 2 and j < 2)
          is_edge = not is_corner and (i < 2 or j < 2 or i >= rows - 2 or j >= cols - 2)

          if is_corner:
            # top left corner
            if i < 2 and j < 2:
              if i == 0 and j == 0:
                  row_slice_map = slice(i, i + 3)
                  col_slice_map = slice(j, j + 3)
                  mask_slice = (slice(2, 5), slice(2, 5))
              elif i == 0 and j == 1:
                  row_slice_map = slice(i, i + 3)
                  col_slice_map = slice(j - 1, j + 3)
                  mask_slice = (slice(2, 5), slice(1, 5))
              elif i == 1 and j == 0:
                  row_slice_map = slice(i - 1, i + 3)
                  col_slice_map = slice(j, j + 3)
                  mask_slice = (slice(1, 5), slice(2, 5))
              elif i == 1 and j == 1:
                  row_slice_map = slice(i - 1, i + 3)
                  col_slice_map = slice(j - 1, j + 3)
                  mask_slice = (slice(1, 5), slice(1, 5))

            # top right corner
            elif i < 2 and j >= cols - 2:
              if i == 0 and j == cols - 1:
                  row_slice_map = slice(i, i + 3)
                  col_slice_map = slice(cols - 3, cols)
                  mask_slice = (slice(2, 5), slice(0, 3))
              elif i == 0 and j == cols - 2:
                  row_slice_map = slice(i, i + 3)
                  col_slice_map = slice(cols - 4, cols)
                  mask_slice = (slice(2, 5), slice(0, 4))
              elif i == 1 and j == cols - 1:
                  row_slice_map = slice(i - 1, i + 3)
                  col_slice_map = slice(cols - 3, cols)
                  mask_slice = (slice(1, 5), slice(0, 3))
              elif i == 1 and j == cols - 2:
                  row_slice_map = slice(i - 1, i + 3)
                  col_slice_map = slice(cols - 4, cols)
                  mask_slice = (slice(1, 5), slice(0, 4))

            # bottom right corner
            elif i >= rows - 2 and j >= cols - 2:
              if i == rows - 1 and j == cols - 1:
                  row_slice_map = slice(rows - 3, rows)
                  col_slice_map = slice(cols - 3, cols)
                  mask_slice = (slice(0, 3), slice(0, 3))
              elif i == rows - 1 and j == cols - 2:
                  row_slice_map = slice(rows - 3, rows)
                  col_slice_map = slice(cols - 4, cols)
                  mask_slice = (slice(0, 3), slice(0, 4))
              elif i == rows - 2 and j == cols - 1:
                  row_slice_map = slice(rows - 4, rows)
                  col_slice_map = slice(cols - 3, cols)
                  mask_slice = (slice(0, 4), slice(0, 3))
              elif i == rows - 2 and j == cols - 2:
                  row_slice_map = slice(rows - 4, rows)
                  col_slice_map = slice(cols - 4, cols)
                  mask_slice = (slice(0, 4), slice(0, 4))

            # bottom left corner
            elif i >= rows - 2 and j < 2:
              if i == rows - 1 and j == 0:
                  row_slice_map = slice(rows - 3, rows)
                  col_slice_map = slice(j, j + 3)
                  mask_slice = (slice(0, 3), slice(2, 5))
              elif i == rows - 1 and j == 1:
                  row_slice_map = slice(rows - 3, rows)
                  col_slice_map = slice(j - 1, j + 3)
                  mask_slice = (slice(0, 3), slice(1, 5))
              elif i == rows - 2 and j == 0:
                  row_slice_map = slice(rows - 4, rows)
                  col_slice_map = slice(j, j + 3)
                  mask_slice = (slice(0, 4), slice(2, 5))
              elif i == rows - 2 and j == 1:
                  row_slice_map = slice(rows - 4, rows)
                  col_slice_map = slice(j - 1, j + 3)
                  mask_slice = (slice(0, 4), slice(1, 5))

          elif is_edge:
            # top edge
            if i < 2:
              if i == 0:
                  row_slice_map = slice(i, i + 3)
                  col_slice_map = slice(j - 2, j + 3)
                  mask_slice = (slice(2, 5), slice(0, 5))
              elif i == 1:
                  row_slice_map = slice(i - 1, i + 3)
                  col_slice_map = slice(j - 2, j + 3)
                  mask_slice = (slice(1, 5), slice(0, 5))

            # left edge
            elif j < 2:
              if j == 0:
                  row_slice_map = slice(i - 2, i + 3)
                  col_slice_map = slice(j, j + 3)
                  mask_slice = (slice(0, 5), slice(2, 5))
              elif j == 1:
                  row_slice_map = slice(i - 2, i + 3)
                  col_slice_map = slice(j - 1, j + 3)
                  mask_slice = (slice(0, 5), slice(1, 5))

            # bottom edge
            elif i >= rows - 2:
              if i == rows - 1:
                  row_slice_map = slice(rows - 3, rows)
                  col_slice_map = slice(j - 2, j + 3)
                  mask_slice = (slice(0, 3), slice(0, 5))
              elif i == rows - 2:
                  row_slice_map = slice(rows - 4, rows)
                  col_slice_map = slice(j - 2, j + 3)
                  mask_slice = (slice(0, 4), slice(0, 5))

            # right edge
            elif j >= cols - 2:
              if j == cols - 1:
                  row_slice_map = slice(i - 2, i + 3)
                  col_slice_map = slice(cols - 3, cols)
                  mask_slice = (slice(0, 5), slice(0, 3))
              elif j == cols - 2:
                  row_slice_map = slice(i - 2, i + 3)
                  col_slice_map = slice(cols - 4, cols)
                  mask_slice = (slice(0, 5), slice(0, 4))

          else:
            # internal points
            row_slice_map = slice(i - 2, i + 3)
            col_slice_map = slice(j - 2, j + 3)
            mask_slice = (slice(0, 5), slice(0, 5))

          # value for the pixel
          local, val = get_local_and_val_2d_map(
              output_image[k, :, :], map_array, mask_array,
              row_slice_map, col_slice_map, mask_slice
          )

          # update the value
          output_image[k, i, j] = val


  output_image[np.isnan(output_image) | np.isinf(output_image)] = -1

  valid_pixels = output_image[output_image != -1]
  if valid_pixels.size > 0:
    mean_val = np.mean(valid_pixels)
  else:
    mean_val = 0

  output_image[output_image == -1] = mean_val

  return output_image

In [ ]:
# Source: MATLAB code provided by James Day
def map_mask_simple_bright(image_bin, threshold):
  value = np.mean(image_bin, axis=0)
  value_test = np.mean(value)
  threshold_up = value_test * threshold
  maskmap = (image_bin > threshold_up).astype(np.uint8)
  return maskmap

In [ ]:
# Source: MATLAB code provided by James Day
def mask_cycle(image, threshA, threshB):
  mask_array = np.zeros((5,5))

  for i in range(5):
    for j in range(5):
      if i == 2 and j == 2:
        mask_array[i, j] = 0
      else:
        mask_array[i, j] = 1/np.sqrt((i - 2)**2 + (j - 2)**2)

  count = 0
  n_mode = 0

  image_mask = map_mask_simple(image, 1000)
  gain_temp_start = put_on_mask_2d(image, image_mask, mask_array)

  values, counts = np.unique(gain_temp_start, return_counts=True)
  mode_value = values[np.argmax(counts)]
  baseline = np.sum(gain_temp_start == mode_value)

  cutoff = baseline + 1

  threshold_up = 1000
  threshold_down = 0

  for i in range(20):
    image_mask = map_mask_simple(gain_temp_start, threshA)
    gain_temp = put_on_mask_2d(gain_temp_start, image_mask, mask_array)

    values, counts = np.unique(gain_temp, return_counts=True)
    mode_value = values[np.argmax(counts)]
    BB = np.sum(gain_temp == mode_value)

    if BB > cutoff:
      threshold_down = threshA
      threshA = (threshold_down+threshold_up)/2
    elif BB == cutoff:
      image_mask = map_mask_simple(gain_temp_start, threshA)
      gain_temp = put_on_mask_2d(gain_temp_start, image_mask, mask_array)
      break
    elif BB<cutoff:
      threshold_up = threshA
      threshA = (threshold_down+threshold_up)/2

    if (threshold_up-threshold_down)*2/(threshold_up+threshold_down)*100 < 1:
      threshA = threshold_up*1.05
      image_mask = map_mask_simple(gain_temp_start, threshA)
      gain_temp = put_on_mask_2d(gain_temp_start, image_mask, mask_array)
      break

  threshA = threshold_up*1.5
  image_mask = map_mask_simple(gain_temp_start, threshA)
  gain_temp = put_on_mask_2d(gain_temp_start, image_mask, mask_array)

  image_mask = map_mask_simple_bright(gain_temp, 1000)
  gain_temp_start = put_on_mask_3d(gain_temp, image_mask, mask_array)

  values, counts = np.unique(gain_temp_start, return_counts=True)
  mode_value = values[np.argmax(counts)]
  baseline = np.sum(gain_temp_start == mode_value)

  cutoff = baseline + 1
  threshold_up = 1000
  threshold_down = 0

  for i in range(20):
    image_mask = map_mask_simple_bright(gain_temp_start, threshB)
    gain_temp = put_on_mask_3d(gain_temp_start, image_mask, mask_array)

    values, counts = np.unique(gain_temp, return_counts=True)
    mode_value = values[np.argmax(counts)]
    BB = np.sum(gain_temp == mode_value)

    if BB>cutoff:
      threshold_down = threshB
      threshB = (threshold_down+threshold_up)/2
    elif BB==cutoff:
      image_mask = map_mask_simple_bright(gain_temp_start, threshB)
      gain_temp = put_on_mask_3d(gain_temp_start, image_mask, mask_array)
      break
    elif BB<cutoff:
      threshold_up = threshB
      threshB = (threshold_down+threshold_up)/2

    if (threshold_up-threshold_down)*2/(threshold_up+threshold_down)*100 < 1:
      break

    threshB = threshold_up*1.5
    image_mask = map_mask_simple_bright(gain_temp_start, threshB)
    gain_temp = put_on_mask_3d(gain_temp_start, image_mask, mask_array)

  return gain_temp

In [ ]:
# Source: MATLAB code provided by James Day
def map_mask_simple_bright2(image, threshold_up, threshold_down):
  maskmap = np.isnan(image) | np.isinf(image)
  maskmap |= (image > threshold_up) | (image < threshold_down)
  return maskmap.astype(np.uint8)

In [ ]:
# Source: MATLAB code provided by James Day
def map_mask_simple2(image, threshold):
  value = np.std(image, axis=0, ddof=0) / np.mean(image, axis=0) * 100
  maskmap = (value > threshold).astype(np.uint8)
  return maskmap

In [ ]:
# Source: MATLAB code provided by James Day
def mask_cycle2(image, threshA, thresh_up, thresh_down):
    # create the 5x5 mask
    mask_array = np.zeros((5,5))
    for i in range(5):
        for j in range(5):
            if i == 2 and j == 2:
                mask_array[i, j] = 0
            else:
                mask_array[i, j] = 1 / np.sqrt((i - 2)**2 + (j - 2)**2)

    # initial mask
    image_mask = map_mask_simple(image, 1000)
    gain_temp_start = put_on_mask_2d(image, image_mask, mask_array)

    # compute baseline & cutoff
    values, counts = np.unique(gain_temp_start, return_counts=True)
    mode_value = values[np.argmax(counts)]
    baseline = np.sum(gain_temp_start == mode_value)
    cutoff = baseline + 1

    t_up, t_down = 1000, 0
    for _ in range(20):
        image_mask = map_mask_simple2(gain_temp_start, threshA)
        gain_temp_A = put_on_mask_2d(gain_temp_start, image_mask, mask_array)

        values, counts = np.unique(gain_temp_A, return_counts=True)
        mode_value = values[np.argmax(counts)]
        BB = np.sum(gain_temp_A == mode_value)

        if BB > cutoff:
            t_down = threshA
            threshA = (t_down + t_up) / 2
        elif BB == cutoff:
            image_mask = map_mask_simple2(gain_temp_start, threshA)
            gain_temp_A = put_on_mask_2d(gain_temp_start, image_mask, mask_array)
            break
        else:
            t_up = threshA
            threshA = (t_down + t_up) / 2

        if (t_up - t_down)*2/(t_up + t_down)*100 < 1:
            break

    threshA = t_up * 1.25
    image_mask = map_mask_simple(gain_temp_start, threshA)
    gain_temp = put_on_mask_2d(gain_temp_start, image_mask, mask_array)

    values, counts = np.unique(gain_temp_A, return_counts=True)
    mode_value = values[np.argmax(counts)]
    baseline = np.sum(gain_temp_A == mode_value)
    cutoff = baseline + 1

    t_up, t_down = 100, 1
    for _ in range(20):
        image_mask = map_mask_simple_bright2(gain_temp, thresh_up, thresh_down)
        gain_temp_B = put_on_mask_3d(gain_temp, image_mask, mask_array)

        values, counts = np.unique(gain_temp_B, return_counts=True)
        mode_value = values[np.argmax(counts)]
        BB = np.sum(gain_temp_B == mode_value)

        if BB > cutoff:
            t_down = thresh_up
            thresh_up = (t_down + t_up) / 2
        elif BB == cutoff:
            break
        else:
            t_up = thresh_up
            thresh_up = (t_down + t_up) / 2

        if (t_up - t_down)*2/(t_up + t_down)*100 < 1:
            break

    thresh_up = t_up * 1.25
    image_mask = map_mask_simple_bright2(gain_temp, thresh_up, thresh_down)
    gain_temp_B = put_on_mask_3d(gain_temp, image_mask, mask_array)

    values, counts = np.unique(gain_temp_B, return_counts=True)
    mode_value = values[np.argmax(counts)]
    baseline = np.sum(gain_temp_B == mode_value)
    cutoff = baseline + 1

    t_up, t_down = 10, 0
    for _ in range(20):
        image_mask = map_mask_simple_bright2(gain_temp_B, t_up, t_down)
        gain_temp = put_on_mask_3d(gain_temp_B, image_mask, mask_array)

        values, counts = np.unique(gain_temp, return_counts=True)
        mode_value = values[np.argmax(counts)]
        BB = np.sum(gain_temp == mode_value)

        if BB > cutoff:
            t_up = t_down
            t_down = (t_down + t_up) / 2
        elif BB == cutoff:
            image_mask = map_mask_simple_bright2(gain_temp_B, t_up, t_down)
            gain_temp = put_on_mask_3d(gain_temp_B, image_mask, mask_array)
            break
        else:
            t_down = t_down
            t_down = (t_down + t_up) / 2

        if (t_up - t_down)*2/(t_up + t_down)*100 < 1:
            t_down = t_down
            image_mask = map_mask_simple_bright2(gain_temp_B, t_up, t_down)
            gain_temp = put_on_mask_3d(gain_temp_B, image_mask, mask_array)
            break

    t_down = t_down - t_down*0.1
    image_mask = map_mask_simple_bright2(gain_temp_B, t_up, t_down)
    gain_temp = put_on_mask_3d(gain_temp_B, image_mask, mask_array)

    return gain_temp


In [ ]:
# Source: MATLAB code provided by James Day
def map_mask_sinogram(image_bin, threshold):
    image_perc = image_bin.copy()
    angles, rows, cols = image_perc.shape

    if rows > 4:
        j_mid = np.arange(2, rows - 2)
        windows_mid = np.stack([image_perc[:, j_mid-2, :],
                                image_perc[:, j_mid-1, :],
                                image_perc[:, j_mid, :],
                                image_perc[:, j_mid+1, :],
                                image_perc[:, j_mid+2, :]], axis=-1)
        means_mid = np.mean(windows_mid, axis=-1)
        stds_mid = np.std(windows_mid, axis=-1, ddof=0)
        tdown_mid = means_mid - stds_mid * threshold
        tup_mid = means_mid + stds_mid * threshold

        mask_mid = (image_perc[:, j_mid, :] < tdown_mid) | (image_perc[:, j_mid, :] > tup_mid)

        weighted_mid = (windows_mid[...,0]*1/4 + windows_mid[...,1] + windows_mid[...,3] + windows_mid[...,4]*1/4) / (1 + 1 + 1/4 + 1/4)
        image_perc[:, j_mid, :][mask_mid] = weighted_mid[mask_mid]

    # handling edges
    for j in [0, 1, rows-2, rows-1]:
        if j == 0:
            test = image_perc[:, j:j+5, :]
            means = np.mean(test, axis=1)
            stds = np.std(test, axis=1, ddof=0)
            tdown = means - stds * threshold
            tup = means + stds * threshold
            mask = (image_perc[:, j, :] < tdown) | (image_perc[:, j, :] > tup)
            weighted = (test[:,1,:] + test[:,2,:]*1/4 + test[:,3,:]*1/9 + test[:,4,:]*1/16) / (1 + 1/4 + 1/9 + 1/16)
            image_perc[:, j, :][mask] = weighted[mask]

        elif j == 1:
            test = image_perc[:, j-1:j+4, :]
            means = np.mean(test, axis=1)
            stds = np.std(test, axis=1, ddof=0)
            tdown = means - stds * threshold
            tup = means + stds * threshold
            mask = (image_perc[:, j, :] < tdown) | (image_perc[:, j, :] > tup)
            weighted = (test[:,0,:] + test[:,2,:] + test[:,3,:]*1/4 + test[:,4,:]*1/9) / (1 + 1 + 1/4 + 1/9)
            image_perc[:, j, :][mask] = weighted[mask]

        elif j == rows-2:
            test = image_perc[:, j-3:j+2, :]
            means = np.mean(test, axis=1)
            stds = np.std(test, axis=1, ddof=0)
            tdown = means - stds * threshold
            tup = means + stds * threshold
            mask = (image_perc[:, j, :] < tdown) | (image_perc[:, j, :] > tup)
            weighted = (test[:,0,:]*1/9 + test[:,1,:]*1/4 + test[:,2,:] + test[:,4,:]) / (1 + 1 + 1/4 + 1/9)
            image_perc[:, j, :][mask] = weighted[mask]

        elif j == rows-1:
            test = image_perc[:, j-4:j+1, :]
            means = np.mean(test, axis=1)
            stds = np.std(test, axis=1, ddof=0)
            tdown = means - stds * threshold
            tup = means + stds * threshold
            mask = (image_perc[:, j, :] < tdown) | (image_perc[:, j, :] > tup)
            weighted = (test[:,0,:]*1/16 + test[:,1,:]*1/9 + test[:,2,:]*1/4 + test[:,3,:]) / (1 + 1/4 + 1/9 + 1/16)
            image_perc[:, j, :][mask] = weighted[mask]

    return image_perc


In [ ]:
# Source: MATLAB code provided by James Day
def process_log(input_array, mod):
  output = np.zeros_like(input_array, dtype=float)
  valid_mask = (input_array > 0) & np.isfinite(input_array)
  output[valid_mask] = -np.log(input_array[valid_mask] / mod)

  return output

In [ ]:
bin1_blank, bin2_blank, bin3_blank, bin4_blank, bin5_blank, bin6_blank, bin_all_blank = load_sinograms(FILENAME_BASE_BLANK, imb)

# apply a mask to each image that was scanned
for i in range(start_acquisition, end_aqusition):

  bin1_object, bin2_object, bin3_object, bin4_object, bin5_object, bin6_object, bin_all_object = load_sinograms(FILENAME_BASE, i)

  image1_background = apply_corrections(bin1_blank,mask1,gain1)
  image2_background = apply_corrections(bin2_blank,mask2,gain2)
  image3_background = apply_corrections(bin3_blank,mask3,gain3)
  image4_background = apply_corrections(bin4_blank,mask4,gain4)
  image5_background = apply_corrections(bin5_blank,mask5,gain5)
  image6_background = apply_corrections(bin6_blank,mask6,gain6)
  imageall_background = apply_corrections(bin_all_blank,mask_all,gain_all)


  image1_object = apply_corrections(bin1_object,mask1,gain1)
  image2_object = apply_corrections(bin2_object,mask2,gain2)
  image3_object = apply_corrections(bin3_object,mask3,gain3)
  image4_object = apply_corrections(bin4_object,mask4,gain4)
  image5_object = apply_corrections(bin5_object,mask5,gain5)
  image6_object = apply_corrections(bin6_object,mask6,gain6)
  imageall_object = apply_corrections(bin_all_object,mask_all,gain_all)

  backgrounds = [image1_background, image2_background, image3_background, image4_background, image5_background, image6_background, imageall_background]
  objects = [image1_object, image2_object, image3_object, image4_object, image5_object, image6_object, imageall_object]

  threshA = 10
  threshB = 4

  vount = 1

  for j in range(7):
    test_img_obj = mask_cycle(objects[j],threshA,threshB)
    test_img_bk = mask_cycle(backgrounds[j],threshA,threshB)

    image_bin_proj = test_img_obj/test_img_bk

    test_img_proj = mask_cycle2(image_bin_proj,85,1.05,0.07)

    test_img_proj_fin = map_mask_sinogram(process_log(test_img_proj,1),1.5)

    if j<6:
      np.save(FILENAME_BASE+f"Masked_Sinograms/{chr(i+65)}_bin{j+1}_projection.npy", test_img_proj_fin)
    else:
      np.save(FILENAME_BASE+f"Masked_Sinograms/{chr(i+65)}_bin_all_projection.npy", test_img_proj_fin)